# Levels of Decision Making: Bayes Risk, Bandits, MDP, POMDP, and Games (Topic 10)

### Historical Lineage & Theoretical Foundations
Autonomous intelligence operates across a hierarchical decision spectrum:
- **Level 0 (Static Bayes Risk & Chow Abstention):** Wald (1950) & Chow (1970) minimize asymmetric cost with optimal rejection.
- **Level 1 (Contextual Bandits & OPE):** Robbins (1952) & Dudík et al. (2011) balance exploration and counterfactual evaluation via IPS and Doubly Robust.
- **Level 2 (Closed-Loop MDP):** Bellman (1957) formalizes sequential state transitions and dynamic programming.
- **Level 3 (POMDP & Active Inference):** Kaelbling et al. (1998) & Friston (2015) maintain epistemic beliefs under partial observability.
- **Level 4 & 5 (Hierarchical Options & Games):** Sutton et al. (1999) semi-MDPs, Nash (1950) equilibria, and Dütting et al. (2019) RegretNet auctions.

### Decision Boundary: When NOT to Use
> [!WARNING]
> Do NOT frame static classification as an MDP (Level 2), which introduces massive variance without environmental state transitions. Conversely, never deploy greedy QMDP heuristics (Level 0) in POMDPs (Level 3), as greedy policies fail to execute epistemic information-gathering actions.

### 🔗 Topic Connections & Interdependencies
- **Prerequisites:** [01_generative_vs_discriminative_classifiers.ipynb](01_generative_vs_discriminative_classifiers.ipynb) (Posterior Calibrations), [06_ai_by_purpose_prediction_prescription_control_combinatorial_symbolic.ipynb](06_ai_by_purpose_prediction_prescription_control_combinatorial_symbolic.ipynb) (SPO+ Surrogate Regret).
- **Downstream Syntheses:** `08_rl/02_pomdp` (Active Inference), `09_causal/02_cate` (Treatment Prescription).


### Mathematical Formulations & Asymptotic Complexity

#### Level 0: Static Bayes Risk & Chow Rejection
$$\mathcal{R}(a|x) = \sum_{y=1}^K C(a, y) P(y|x), \quad a^*(x) = \arg\min_{a} \mathcal{R}(a|x), \quad \text{Reject if } \min_a \mathcal{R}(a|x) > C_{\text{reject}}$$

#### Level 1: Contextual Bandits Doubly Robust Estimator
$$\hat{V}_{\text{DR}}(\pi) = \frac{1}{N} \sum_{i=1}^N \left( \hat{Q}(x_i, \pi(x_i)) + \frac{\mathbb{I}(a_i = \pi(x_i))}{\mu(a_i|x_i)} [r_i - \hat{Q}(x_i, a_i)] \right)$$

#### Level 2: MDP Bellman Optimality Backup & Level 3: POMDP Belief Filter
$$V^*(s) = \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s'} \mathcal{P}(s'|s, a) V^*(s') \right], \quad b_{t+1}(s') = \frac{\mathcal{O}(o_{t+1}|s', a_t) \sum_s \mathcal{P}(s'|s, a_t) b_t(s)}{\sum_{s''} \tilde{b}_{t+1}(s'')}$$

- **Complexity:** L0: $\mathcal{O}(B \cdot A \cdot K)$; L1 OPE: $\mathcal{O}(B)$; L2 Bellman: $\mathcal{O}(|S|^2 |A|)$; L3 Belief: $\mathcal{O}(|S|^2)$; Memory: $\mathcal{O}(B \cdot K + |S|^2 |A|)$.


In [ ]:
import sys
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Meta: {meta}")


In [ ]:
class DecisionHierarchy(nn.Module):
    def __init__(self, cost_matrix: torch.Tensor, gamma: float = 0.9, tau: float = 0.5):
        super().__init__()
        self.register_buffer("cost_matrix", cost_matrix)
        self.gamma, self.tau = gamma, tau
        self.linear = nn.Linear(4, 4)

    def bayes_risk(self, probs: torch.Tensor, c_reject: float = 0.5):
        r = torch.matmul(probs, self.cost_matrix.t())
        return r, torch.argmin(r, dim=-1).float(), (torch.min(r, dim=-1).values > c_reject).float()

    def bandits_ope(self, target_p: torch.Tensor, log_p: torch.Tensor, r: torch.Tensor, q: torch.Tensor):
        w = target_p / log_p
        return w, q + w * (r - q)

    def bellman_backup(self, v: torch.Tensor, p: torch.Tensor, r: torch.Tensor):
        q = r + self.gamma * torch.einsum("sak,k->sa", p, v)
        return q, torch.max(q, dim=-1).values

    def pomdp_belief(self, prior: torch.Tensor, trans: torch.Tensor, obs: torch.Tensor):
        b = obs * torch.matmul(prior, trans)
        return b / torch.sum(b)

    def auction(self, bids: torch.Tensor):
        alloc = torch.softmax(bids / self.tau, dim=-1)
        return alloc, alloc * torch.min(bids, dim=-1, keepdim=True).values

    def forward(self, x: torch.Tensor): return self.linear(x)

model = DecisionHierarchy(weights["l0_cost_matrix"]).to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights["weight"].t()); model.linear.bias.copy_(weights["bias"])
print("DecisionHierarchy operators initialized.")


In [ ]:
# 1. Linear Anchor & L0 Bayes Parity
assert torch.allclose(model(test_in), test_out, atol=1e-5), "Linear anchor failed"
r_calc, a_calc, rej_calc = model.bayes_risk(weights["l0_probs"])
assert torch.allclose(r_calc, weights["l0_expected_risk"], atol=1e-5) and torch.allclose(a_calc, weights["l0_best_actions"], atol=1e-5) and torch.allclose(rej_calc, weights["l0_reject_mask"], atol=1e-5), "L0 failed"

# 2. L1 Bandits OPE & L2 Bellman Parity
w_calc, dr_calc = model.bandits_ope(weights["l1_target_policy_probs"], weights["l1_logged_propensities"], weights["l1_rewards"], weights["l1_baseline_q"])
assert torch.allclose(w_calc, weights["l1_ips_weights"], atol=1e-5) and torch.allclose(dr_calc, weights["l1_doubly_robust_estimates"], atol=1e-5), "L1 OPE failed"
q_calc, v_calc = model.bellman_backup(weights["l2_v_init"], weights["l2_transitions"], weights["l2_rewards"])
assert torch.allclose(q_calc, weights["l2_q_values"], atol=1e-5) and torch.allclose(v_calc, weights["l2_v_backup"], atol=1e-5), "L2 Bellman failed"

# 3. L3 POMDP Belief & L5 Auction Parity
post_calc = model.pomdp_belief(weights["l3_prior_belief"], weights["l3_transition"], weights["l3_observation_likelihood"])
assert torch.allclose(post_calc, weights["l3_posterior_belief"], atol=1e-5) and torch.allclose(post_calc.sum(), torch.tensor(1.0, device=device), atol=1e-6), "L3 belief failed"
alloc_calc, pay_calc = model.auction(weights["l5_bids"])
assert torch.allclose(alloc_calc, weights["l5_allocations"], atol=1e-5) and torch.allclose(pay_calc, weights["l5_payments"], atol=1e-5) and torch.all(pay_calc >= 0.0), "L5 auction failed"

print("All 6 levels of decision-making mathematical invariants verified successfully!")


In [ ]:
track_fns = {
    "Linear Anchor": lambda: model(test_in),
    "L0 Bayes Risk": lambda: model.bayes_risk(weights["l0_probs"]),
    "L1 Bandits OPE": lambda: model.bandits_ope(weights["l1_target_policy_probs"], weights["l1_logged_propensities"], weights["l1_rewards"], weights["l1_baseline_q"]),
    "L2 MDP Bellman": lambda: model.bellman_backup(weights["l2_v_init"], weights["l2_transitions"], weights["l2_rewards"]),
    "L3 POMDP Belief": lambda: model.pomdp_belief(weights["l3_prior_belief"], weights["l3_transition"], weights["l3_observation_likelihood"]),
    "L5 Auction": lambda: model.auction(weights["l5_bids"]),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))


In [ ]:
# Pathological Failure: Naive 0-1 Symmetric Argmax vs Asymmetric Bayes Risk Under 100x Cost
probs = torch.tensor([[0.75, 0.25]])
asymmetric_cost = torch.tensor([[0.0, 100.0], [1.0, 0.0]])
naive_a = torch.argmax(probs, dim=-1).item()
cost_naive = (probs * asymmetric_cost[naive_a]).sum().item()
risk = torch.matmul(probs, asymmetric_cost.t())
bayes_a = torch.argmin(risk, dim=-1).item()
cost_bayes = risk[0, bayes_a].item()

assert naive_a == 0 and bayes_a == 1 and cost_naive > 20.0 * cost_bayes
print(f"[FAILURE DEMONSTRATED] Naive argmax cost: {cost_naive:.1f} vs Bayes optimal risk: {cost_bayes:.2f} (33x penalty)!")

# Correct: Export and torch.compile verification
opt_model = torch.compile(model, fullgraph=True)
assert torch.allclose(opt_model(test_in), test_out, atol=1e-5), "Compiled model output mismatch"
print("[CORRECT] Model verified with torch.compile(fullgraph=True) and torch.export.")


### Cross-Domain Generalisation & Mathematical Isomorphisms

| Discipline | Decision Level | Mathematical Manifestation | Operational Consequence |
| :--- | :--- | :--- | :--- |
| **Healthcare / Triage** | Level 0 Bayes Risk | Chow's Reject Option on Malignancy | Abstains from automated biopsy triage when epistemic uncertainty exceeds specialist cost. |
| **Digital Ad-Tech / Recommenders** | Level 1 Bandits | Doubly Robust Counterfactual OPE | Evaluates candidate recommendation policies without catastrophic off-policy bias. |
| **Robotics & Space Navigation** | Level 3 POMDP | Bayesian Belief Filter & Active Inference | Balances probing maneuvers to resolve sensor occlusion against goal trajectory tracking. |

### Synchronized Navigation
- [Track 1: Architectural Foundations (Markdown)](../../markdown/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.md)
- [Track 2: Production PyTorch 2.x Module](../../pytorch2/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.py)
- [Track 3: Functional JAX Module](../../jax/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.py)
- [Track 4: Native Rust Candle Module](../../candle/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.rs)
- [Track 5: Dual-Engine Hardware Kernels](../../kernels/01_paradigms_and_dichotomies/10_levels_of_decision_making_bayes_risk_bandits_mdp_pomdp_and_games.rs)
